# Credit Card Default Prediction - Exploratory Data Analysis

This notebook explores the Default of Credit Card Clients dataset.  
The goal is to understand the structure of the data, target distribution, feature behavior, and relationships before building machine learning models.

### Import libraries ###

In [ ]:
import pandas as pd
import numpy as np

try:
    import matplotlib.pyplot as plt
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "matplotlib"])
    import matplotlib.pyplot as plt
import seaborn as sns
import sys, subprocess

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

ModuleNotFoundError: No module named 'matplotlib'

### Load dataset ###

In [ ]:
from pathlib import Path

DATA_PATH = "../data/raw/default_of_credit_card_clients.csv"
data_path = Path(DATA_PATH)

if not data_path.is_file():
    for base in [Path.cwd(), *Path.cwd().parents]:
        print("base",base)
        print("path",Path.cwd(), *Path.cwd().parents)
        candidate = (base / DATA_PATH).resolve()
        if candidate.is_file():
            data_path = candidate
            break
    else:
        raise FileNotFoundError(f"Dataset not found: {DATA_PATH}")
df = pd.read_csv(data_path, header = 1)
df.head()

In [ ]:
print(f"Data shape: {df.shape}\n")
print(f"Data description: {df.describe().T}")

### Clean column names ###

In [ ]:
df.columns = df.columns.str.strip()
target_col = 'default payment next month'
print(target_col in df.columns)
print(df.columns)

In [ ]:
df.columns.tolist()

### Check missing values ###

In [ ]:
missing_values = df.isnull().sum().sort_values(ascending=False)
missing_values[missing_values > 0]

### checking missing rows ###

In [ ]:
df.duplicated().sum()

In [ ]:
df[target_col].value_counts()

In [ ]:
df[target_col].value_counts(normalize=True)

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x = target_col)
plt.title('Distribution of Default Payment Next Month')
plt.xlabel('Default Payment Next Month')
plt.ylabel('Count')
plt.show()

### Target Distribution Insight

The target variable is imbalanced. Most customers did not default, while a smaller percentage defaulted next month.  
Because of this imbalance, accuracy alone is not enough. We should also evaluate precision, recall, F1-score, and ROC-AUC.

### Rename columns for readability ###

In [ ]:
rename_map = {
    "LIMIT_BAL": "credit_limit",
    "SEX": "sex",
    "EDUCATION": "education",
    "MARRIAGE": "marriage",
    "AGE": "age",
    "PAY_0": "repayment_status_sep",
    "PAY_2": "repayment_status_aug",
    "PAY_3": "repayment_status_jul",
    "PAY_4": "repayment_status_jun",
    "PAY_5": "repayment_status_may",
    "PAY_6": "repayment_status_apr",
    "BILL_AMT1": "bill_amount_sep",
    "BILL_AMT2": "bill_amount_aug",
    "BILL_AMT3": "bill_amount_jul",
    "BILL_AMT4": "bill_amount_jun",
    "BILL_AMT5": "bill_amount_may",
    "BILL_AMT6": "bill_amount_apr",
    "PAY_AMT1": "payment_amount_sep",
    "PAY_AMT2": "payment_amount_aug",
    "PAY_AMT3": "payment_amount_jul",
    "PAY_AMT4": "payment_amount_jun",
    "PAY_AMT5": "payment_amount_may",
    "PAY_AMT6": "payment_amount_apr",
    "default.payment.next.month": "default_next_month"
}

df = df.rename(columns=rename_map)

df.head()

In [ ]:
target_col = "default_next_month"

### Dropping ID Column ###

In [ ]:
if "ID" in df.columns:
    df = df.drop(columns=["ID"])
df.head()

In [ ]:
df = df.rename(columns = {'default payment next month': 'default_next_month'})

### Credit limit analysis ###

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(data=df, x = 'credit_limit', hue = target_col, bins = 40, kde = True)
plt.title("credits_limit distribution by Default status")
plt.xlabel("credits_limit")
plt.ylabel("Count")
plt.show()

In [ ]:
df.groupby(target_col)['credit_limit'].describe()  

### Credit Limit Insight

Credit limit appears to be related to default behavior. Customers with lower credit limits may show different default patterns compared to customers with higher credit limits.

### Age analysis ###

In [ ]:
plt.figure(figsize=(6, 4))
sns.histplot(data=df, x = 'age', hue = target_col, bins = 40, kde = True)

### Categorical feature analysis ###

In [ ]:
plot_df = df.copy()
plot_df[target_col] = plot_df[target_col].map({
    0: 'No Default', 1: 'Default'
    })


In [ ]:
categorical_cols = ['sex', 'education', 'marriage']

for col in categorical_cols:
    plt.figure(figsize=(6, 4))
    sns.countplot(data=plot_df, x = col, hue = target_col)
    plt.title(f"{col.title()} vs Default Status")
    plt.xlabel(col.title())
    plt.ylabel("Count")
    plt.legend(title='default')
    plt.show()

### Bill amount analysis ###

In [ ]:
bill_cols = [col for col in df.columns if col.startswith('bill_amount')]

In [ ]:
df[bill_cols].describe().T

In [ ]:
plt.figure(figsize =(10,6))
df[bill_cols].boxplot()
plt.title("Bill amounts Distribution Across Months")
plt.xticks(rotation=45)
plt.ylabel("Bill Amount")
plt.show()

### Payment amount analysis ###

In [ ]:
payment_cols = [col for col in df.columns if col.startswith('payment_amount')]

In [ ]:
plt.figure(figsize=(10, 5))
df[payment_cols].boxplot()
plt.title("Previous Payment Amount Distribution Across Months")
plt.xticks(rotation=45)
plt.ylabel("Payment Amount")
plt.show()

### Correlation heatmap ###

In [ ]:
plt.figure(figsize=(16, 12))
corr = df.corr(numeric_only=True)

sns.heatmap(corr, annot=True, cmap='coolwarm', center=0)
plt.title("Correlation Heatmap")
plt.show()

### Save cleaned dataset ###

In [ ]:
processed_path = "../data/processed/credit_default_cleaned.csv"

df.to_csv(processed_path, index=False)

print(f"Cleaned dataset saved to: {processed_path}")

## EDA Summary

Key findings:

1. The dataset contains customer credit, demographic, repayment, bill, and payment information.
2. The target variable is imbalanced, with fewer default cases than non-default cases.
3. Repayment status variables appear to be highly relevant for default prediction.
4. Credit limit and payment behavior may also provide useful predictive signals.
5. Accuracy alone is not enough for evaluation because the target is imbalanced.
6. The next step is to build baseline and advanced classification models using Logistic Regression, Decision Tree, Random Forest, Gradient Boosting, and XGBoost.